In [4]:
# CELL 1 — Verify Ollama is ready and qwen2.5:7b is available

import requests
import pandas as pd
import os

# Check Ollama
try:
    resp = requests.get('http://localhost:11434/api/tags', timeout=5)
    models = [m['name'] for m in resp.json().get('models', [])]
    print("✅ Ollama running")
    print(f"Available models: {models}")
    if 'qwen2.5:7b' in models:
        print("✅ qwen2.5:7b ready")
    else:
        print("❌ qwen2.5:7b not found")
        print("Run in terminal: ollama pull qwen2.5:7b")
except Exception as e:
    print(f"❌ Ollama not running: {e}")
    print("Start with: ollama serve")

# Check source file
raw_path = 'youtube_collected_raw.csv'
if os.path.exists(raw_path):
    df = pd.read_csv(raw_path)
    print(f"\n✅ Source file: {len(df):,} rows")
    print(f"Columns: {df.columns.tolist()}")
    print(f"Sample text: {df['text'].iloc[0][:80]}")
else:
    print(f"❌ File not found: {raw_path}")

✅ Ollama running
Available models: ['qwen2.5:7b']
✅ qwen2.5:7b ready

✅ Source file: 191,000 rows
Columns: ['text', 'tentative_label', 'label_source', 'category', 'query', 'video_id', 'likes', 'published_at', 'reply_count', 'author_channel', 'title', 'channel', 'video_views', 'video_comment_count', 'video_published_at', 'transcript', 'is_banglish', 'super_category', 'has_emoticon', 'text_length', 'word_count']
Sample text: দেখুন ১বছর প্রতিক্ষার পর আরমান আলিফের এই ঈদে (২০২২) একমাত্র নতুন গানের মিউজিক ভি


C:\Users\user1.CSE-12B21L-LD19\AppData\Local\Temp\ipykernel_47844\2514095746.py:25: DtypeWarning: Columns (16,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(raw_path)


In [5]:
# CELL 2 — Filter raw corpus before annotation
# Remove spam, keep only quality Bengali comments
# This reduces 191k to a cleaner set worth annotating

import pandas as pd
import re
import os

raw_path  = 'youtube_collected_raw.csv'
save_path = 'youtube_for_annotation.csv'

df = pd.read_csv(raw_path)
print(f"Raw: {len(df):,} rows")

# Remove already-annotated 1000 samples
gold = pd.read_csv('annotation_batch_final.csv')
gold_texts = set(gold['text'].astype(str).tolist())
df = df[~df['text'].astype(str).isin(gold_texts)].copy()
print(f"After removing gold 1000: {len(df):,} rows")

# Quality filters
def emoticon_spam(text):
    t = str(text)
    total = len(t.replace(' ',''))
    if total == 0: return True
    emoji_chars = len(re.findall(
        r'[\U0001F300-\U0001FFFF]', t))
    return (emoji_chars / total) > 0.30

def rep_spam(text):
    return bool(re.search(r'(.)\1{6,}', str(text)))

def is_bengali(text, min_chars=8):
    return len(re.findall(
        r'[\u0980-\u09FF]', str(text))) >= min_chars

def word_count(text):
    return len(str(text).split())

df['wc'] = df['text'].apply(word_count)

f1 = ~df['text'].apply(emoticon_spam)
f2 = ~df['text'].apply(rep_spam)
f3 =  df['text'].apply(is_bengali)
f4 =  df['wc'] >= 4
f5 =  df['wc'] <= 150
f6 = ~df.duplicated(subset=['text'], keep='first')

df_clean = df[f1&f2&f3&f4&f5&f6].copy().reset_index(drop=True)
print(f"After quality filters: {len(df_clean):,} rows")

df_clean.to_csv(save_path, index=False)
print(f"✅ Saved: {save_path}")
print(f"\nEstimated annotation time:")
print(f"  At 2 sec/comment: {len(df_clean)*2/3600:.1f} hours")
print(f"  At 5 sec/comment: {len(df_clean)*5/3600:.1f} hours")

C:\Users\user1.CSE-12B21L-LD19\AppData\Local\Temp\ipykernel_47844\2732577994.py:12: DtypeWarning: Columns (16,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(raw_path)


Raw: 191,000 rows
After removing gold 1000: 190,018 rows
After quality filters: 178,912 rows
✅ Saved: youtube_for_annotation.csv

Estimated annotation time:
  At 2 sec/comment: 99.4 hours
  At 5 sec/comment: 248.5 hours


In [6]:
# CELL 3 — Full 190k LLM annotation with resume support
# Runs overnight — auto-saves every 500 comments
# Can be stopped and resumed any time

import requests
import pandas as pd
import time
import os
import json
from datetime import datetime

OLLAMA_URL   = 'http://localhost:11434/api/generate'
CHOSEN_MODEL = 'qwen2.5:7b'
SOURCE_PATH  = 'youtube_for_annotation.csv'
SAVE_PATH    = 'youtube_llm_annotated.csv'
LOG_PATH     = 'annotation_log.txt'

CHECKPOINT_EVERY = 500
CONFIDENCE_THRESHOLD = 0.75  # keep only high confidence

# Load source
df_source = pd.read_csv(SOURCE_PATH)
print(f"Total to annotate: {len(df_source):,}")

# Resume support
if os.path.exists(SAVE_PATH):
    df_done   = pd.read_csv(SAVE_PATH)
    done_texts= set(df_done['text'].astype(str).tolist())
    RESULTS   = df_done.to_dict('records')
    print(f"Resuming: {len(done_texts):,} already done")
    print(f"Remaining: {len(df_source)-len(done_texts):,}")
else:
    done_texts = set()
    RESULTS    = []
    print("Starting fresh annotation run")

def annotate_single(text):
    """
    Returns: (label, confidence_str)
    label = 1 (depressive), 0 (non-depressive), -1 (failed)
    confidence_str = 'HIGH' or 'LOW' or 'FAILED'
    """
    prompt = f"""You are a mental health annotation expert for Bengali social media text.

TASK: Classify this Bengali comment as DEPRESSIVE or NON_DEPRESSIVE.

RULES:
- DEPRESSIVE: Person expresses personal sadness, hopelessness, loneliness, emotional pain, suicidal thoughts, or inability to cope with their own life.
- NON_DEPRESSIVE: Happy, neutral, commenting on a song or video, general opinions, no personal distress expressed.

CRITICAL DISTINCTION:
"এই গানটা অনেক কষ্টের" (This song is very sad) → NON_DEPRESSIVE (about the song)
"আমার অনেক কষ্ট হচ্ছে" (I am in a lot of pain) → DEPRESSIVE (personal)
"আমি মরে যেতে চাই" (I want to die) → DEPRESSIVE
"গানটা ভালো লাগলো" (I liked the song) → NON_DEPRESSIVE

COMMENT: {text}

Reply in this exact format:
LABEL: [DEPRESSIVE or NON_DEPRESSIVE]
CONFIDENCE: [HIGH or LOW]"""

    try:
        response = requests.post(
            OLLAMA_URL,
            json={
                'model':  CHOSEN_MODEL,
                'prompt': prompt,
                'stream': False,
                'options': {
                    'temperature': 0.0,
                    'num_predict': 30,
                }
            },
            timeout=120
        )
        reply = response.json()['response'].strip().upper()

        # Parse label
        if 'NON_DEPRESSIVE' in reply or 'NON-DEPRESSIVE' in reply:
            label = 0
        elif 'DEPRESSIVE' in reply:
            label = 1
        else:
            return -1, 'FAILED'

        # Parse confidence
        if 'HIGH' in reply:
            conf = 'HIGH'
        elif 'LOW' in reply:
            conf = 'LOW'
        else:
            conf = 'LOW'

        return label, conf

    except Exception as e:
        return -1, 'FAILED'

# Main annotation loop
start_time  = time.time()
session_done = 0

print(f"\nStarting at: {datetime.now().strftime('%H:%M:%S')}")
print(f"Model: {CHOSEN_MODEL}")
print(f"Checkpoint every: {CHECKPOINT_EVERY} comments")
print("="*55)

for idx, row in df_source.iterrows():
    text = str(row['text']).strip()

    # Skip if already done
    if text in done_texts:
        continue

    # Annotate
    label, conf = annotate_single(text)

    # Store result
    RESULTS.append({
        'text':           text,
        'ai_label':       label,
        'ai_confidence':  conf,
        'tentative_label':row.get('tentative_label', -1),
        'category':       row.get('category', ''),
        'video_id':       row.get('video_id', ''),
        'has_emoticon':   row.get('has_emoticon', False),
        'is_banglish':    row.get('is_banglish', False),
    })
    done_texts.add(text)
    session_done += 1

    # Progress report every 100
    if session_done % 100 == 0:
        elapsed   = time.time() - start_time
        per_item  = elapsed / session_done
        remaining = (len(df_source) - len(done_texts)) * per_item
        failed    = sum(1 for r in RESULTS if r['ai_label']==-1)
        high_conf = sum(1 for r in RESULTS
                        if r['ai_confidence']=='HIGH')

        print(f"[{datetime.now().strftime('%H:%M')}] "
              f"Done: {len(done_texts):,}/{len(df_source):,} | "
              f"Session: {session_done:,} | "
              f"ETA: {remaining/3600:.1f}h | "
              f"Failed: {failed} | "
              f"High-conf: {high_conf:,}")

    # Checkpoint save
    if session_done % CHECKPOINT_EVERY == 0:
        pd.DataFrame(RESULTS).to_csv(SAVE_PATH, index=False)
        print(f"  💾 Checkpoint saved at {len(done_texts):,}")

# Final save
pd.DataFrame(RESULTS).to_csv(SAVE_PATH, index=False)

# Summary
total_done = len(RESULTS)
failed     = sum(1 for r in RESULTS if r['ai_label']==-1)
high_conf  = sum(1 for r in RESULTS if r['ai_confidence']=='HIGH')
dep        = sum(1 for r in RESULTS if r['ai_label']==1)
non_dep    = sum(1 for r in RESULTS if r['ai_label']==0)
elapsed    = time.time() - start_time

print(f"\n{'='*55}")
print(f"ANNOTATION COMPLETE")
print(f"{'='*55}")
print(f"Total annotated:   {total_done:,}")
print(f"Failed:            {failed:,} ({100*failed/total_done:.1f}%)")
print(f"High confidence:   {high_conf:,} ({100*high_conf/total_done:.1f}%)")
print(f"Depressive:        {dep:,}")
print(f"Non-depressive:    {non_dep:,}")
print(f"Time taken:        {elapsed/3600:.1f} hours")
print(f"Saved: {SAVE_PATH}")

C:\Users\user1.CSE-12B21L-LD19\AppData\Local\Temp\ipykernel_47844\2328497805.py:22: DtypeWarning: Columns (16,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df_source = pd.read_csv(SOURCE_PATH)


Total to annotate: 178,912
Starting fresh annotation run

Starting at: 17:49:33
Model: qwen2.5:7b
Checkpoint every: 500 comments
[17:53] Done: 100/178,912 | Session: 100 | ETA: 118.6h | Failed: 0 | High-conf: 100
[17:57] Done: 200/178,912 | Session: 200 | ETA: 117.6h | Failed: 0 | High-conf: 197
[18:01] Done: 300/178,912 | Session: 300 | ETA: 117.3h | Failed: 0 | High-conf: 295
[18:05] Done: 400/178,912 | Session: 400 | ETA: 117.0h | Failed: 0 | High-conf: 395
[18:09] Done: 500/178,912 | Session: 500 | ETA: 116.7h | Failed: 0 | High-conf: 492
  💾 Checkpoint saved at 500
[18:13] Done: 600/178,912 | Session: 600 | ETA: 116.5h | Failed: 1 | High-conf: 589
[18:16] Done: 700/178,912 | Session: 700 | ETA: 116.4h | Failed: 1 | High-conf: 688
[18:20] Done: 800/178,912 | Session: 800 | ETA: 116.3h | Failed: 1 | High-conf: 788
[18:24] Done: 900/178,912 | Session: 900 | ETA: 116.2h | Failed: 1 | High-conf: 888
[18:28] Done: 1,000/178,912 | Session: 1,000 | ETA: 116.1h | Failed: 1 | High-conf: 988

KeyboardInterrupt: 

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (2906703498.py, line 93)

In [ ]:
# CELL 5 — Train and compare: Gold-only vs Combined
# This is the key experiment — does pseudo-labeling help?

import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, EarlyStoppingCallback
)
from sklearn.metrics import (
    accuracy_score, f1_score,
    precision_score, recall_score, matthews_corrcoef
)
import transformers.modeling_utils as mu
mu.check_torch_load_is_safe = lambda: None

DEVICE   = torch.device('cuda:0')
SEEDS_5  = [42, 123, 999, 2024, 7]  # 5 seeds (faster than 10)
SAHAJ    = 'neuropark/sahajBERT'

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'f1':  f1_score(labels,preds,average='weighted',zero_division=0),
        'mcc': matthews_corrcoef(labels,preds),
        'acc': accuracy_score(labels,preds),
    }

def run_experiment(train_path, val_path, test_path,
                   experiment_name, seeds):
    results = []
    tok = AutoTokenizer.from_pretrained(SAHAJ)

    def tokenize(b):
        return tok(b['text'],truncation=True,
                   padding='max_length',max_length=128)

    tr = Dataset.from_pandas(
        pd.read_csv(train_path)).map(tokenize,batched=True)
    vl = Dataset.from_pandas(
        pd.read_csv(val_path)).map(tokenize,batched=True)
    te = Dataset.from_pandas(
        pd.read_csv(test_path)).map(tokenize,batched=True)

    for ds in [tr,vl,te]:
        ds.set_format('torch',
            columns=['input_ids','attention_mask','label'])

    for seed in seeds:
        set_seed(seed)
        import gc; gc.collect(); torch.cuda.empty_cache()

        model = AutoModelForSequenceClassification.from_pretrained(
            SAHAJ,num_labels=2,
            ignore_mismatched_sizes=True).to(DEVICE)

        args = TrainingArguments(
            output_dir   = f'./tmp_{experiment_name}_{seed}',
            eval_strategy= 'epoch',
            save_strategy= 'no',
            learning_rate= 2e-5,
            per_device_train_batch_size=32,
            per_device_eval_batch_size =64,
            num_train_epochs=8,
            warmup_ratio=0.1,
            weight_decay=0.01,
            fp16=True,
            dataloader_num_workers=0,
            seed=seed,
            report_to='none',
            load_best_model_at_end=True,
            metric_for_best_model='f1',
            greater_is_better=True,
        )

        trainer = Trainer(
            model=model,args=args,
            train_dataset=tr,eval_dataset=vl,
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(
                early_stopping_patience=3)]
        )
        trainer.train()

        pred_out = trainer.predict(te)
        preds    = np.argmax(pred_out.predictions,axis=1)
        true     = pd.read_csv(test_path)['label'].values
        m        = compute_metrics((pred_out.predictions,true))
        m.update({'experiment':experiment_name,'seed':seed})
        results.append(m)

        print(f"  {experiment_name} seed={seed} → "
              f"F1={m['f1']:.4f} MCC={m['mcc']:.4f}")

        del model,trainer
        torch.cuda.empty_cache()

    del tok
    return results

base = '.'

print("EXPERIMENT A — Gold only (1,000 samples)")
print("="*50)
results_gold = run_experiment(
    f'{base}gold_train.csv',
    f'{base}gold_val.csv',
    f'{base}gold_test.csv',
    'gold_only', SEEDS_5
)

print("\nEXPERIMENT B — Gold + Pseudo-labels (combined)")
print("="*50)
results_combined = run_experiment(
    f'{base}combined_train.csv',
    f'{base}combined_val.csv',
    f'{base}combined_test.csv',
    'combined', SEEDS_5
)

# Compare
from scipy import stats

all_results = results_gold + results_combined
df_res = pd.DataFrame(all_results)
df_res.to_csv(
    'semisupervised_comparison.csv',
    index=False)

gold_f1s = df_res[df_res['experiment']=='gold_only']['f1'].values
comb_f1s = df_res[df_res['experiment']=='combined']['f1'].values

print("\nFINAL COMPARISON")
print("="*50)
print(f"Gold only:  F1={gold_f1s.mean():.4f}±{gold_f1s.std():.4f}")
print(f"Combined:   F1={comb_f1s.mean():.4f}±{comb_f1s.std():.4f}")
print(f"Difference: {comb_f1s.mean()-gold_f1s.mean():+.4f}")

t,p = stats.ttest_rel(comb_f1s, gold_f1s)
print(f"p-value:    {p:.4f} "
      f"({'✅ significant' if p<0.05 else '❌ not significant'})")

if comb_f1s.mean() > gold_f1s.mean():
    print("\n✅ Pseudo-labeling HELPS — include in paper")
else:
    print("\n❌ Pseudo-labeling does not help — report as")
    print("   honest negative result — still publishable")

In [14]:
!pip install emoji

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
# CELL 4 — Build semi-supervised training dataset
# EPH resources/helpers copied verbatim from annotation.ipynb, cell 8.
# Run in a fresh kernel to avoid an earlier notebook's model-loader patches.
%pip install emoji
import re
import emoji

ALL_EMOTION = set([
    'মরতে','মরে','আত্মহত্যা','মৃত্যু','শেষ','বাঁচতে','কষ্ট','যন্ত্রণা',
    'ব্যথা','কাঁদি','কাঁদছি','কাঁদছে','একা','নিঃসঙ্গ','অসহায়','হতাশ',
    'নিরাশ','বিষণ্ণ','ক্লান্ত','বিধ্বস্ত','ভেঙে','ভেঙেছি','দুঃখ','দুঃখী',
    'খারাপ','অস্থির','চিন্তিত','ভয়','রাগ','সুখী','খুশি','আনন্দ','ভালো',
    'ভালোবাসা','আশা','স্বপ্ন','সফল','আলো','হাসি',
    'kosto','kanna','ekla','mara','shesh','nirash','kharap',
    'mon kharap','valo nei','tension','worry','sad','depressed','lonely',
    'happy','joy','love','hope','smile','bhalo',
])
NEGATION = {'না','নেই','নয়','নি','নিই'}
BENGALI_SW = {
    'এবং','ও','বা','কিন্তু','যদি','তবে','কারণ','যে','যা','এই','ওই',
    'সেই','এটা','ওটা','সেটা','আমি','আমার','আমাকে','তুমি','তোমার',
    'সে','তার','তাকে','এখানে','ওখানে','হয়','হবে','করা','করে',
    'থেকে','দিয়ে','জন্য','মধ্যে','আছে','ছিল','কি','কী','কোন',
    'খুব','অনেক','বেশি',
}
SAFE_SW = BENGALI_SW - ALL_EMOTION - NEGATION
EMOTICONS = {
    '😭':'[CRY]','😢':'[SAD_FACE]','😔':'[PENSIVE]','💔':'[BROKEN_HEART]',
    '😩':'[WEARY]','🥺':'[PLEADING]','☠️':'[SKULL]','💀':'[DEATH]',
    '😞':'[DISAPPOINTED]','😟':'[WORRIED]','😓':'[SWEAT_SAD]',
    '🖤':'[BLACK_HEART]','😊':'[SMILE]','😄':'[HAPPY]',
    '❤️':'[HEART]','😂':'[LAUGH]','🙏':'[PRAYER]',
}

# ── Preprocessing ──────────────────────────────────────────
def _rn(text):
    t=re.sub(r'http\S+|www\S+','',str(text))
    t=re.sub(r'@\w+','',t); t=re.sub(r'#\w+','',t)
    t=re.sub(r'<.*?>','',t); return re.sub(r'\s+',' ',t).strip()

def preprocess_P0(t): return str(t).strip()
def preprocess_P1(t): return _rn(t)
def preprocess_P2(t):
    t=_rn(t); t=re.sub(r'[a-zA-Z]','',t)
    t=re.sub(r'[^\u0980-\u09FF\s]','',t)
    return ' '.join(w for w in t.split() if w not in BENGALI_SW)
def preprocess_P3(t):
    t=preprocess_P2(t); t=re.sub(r'\d+','',t)
    return ' '.join(w for w in t.split() if len(w)>=2)
def _base(text,em=True,neg=True,bn=True,el=True):
    t=str(text)
    if em:
        for e,tag in EMOTICONS.items(): t=t.replace(e,f' {tag} ')
        t=emoji.replace_emoji(t,replace='')
    else:
        for e in EMOTICONS: t=t.replace(e,' ')
        t=emoji.replace_emoji(t,replace='')
    t=_rn(t)
    if el: t=re.sub(r'(.)\1{2,}',r'\1\1',t)
    result=[]
    for tok in t.split():
        if em and tok.startswith('[') and tok.endswith(']'):
            result.append(tok); continue
        if re.match(r'^[^\u0980-\u09FFa-zA-Z\[\]]+$',tok): continue
        if tok in NEGATION:
            if neg: result.append(tok)
            elif tok not in SAFE_SW: result.append(tok)
            continue
        if tok in ALL_EMOTION: result.append(tok); continue
        if re.search(r'[a-zA-Z]',tok):
            if bn and len(tok)>=3: result.append(tok.lower())
            continue
        if tok in SAFE_SW: continue
        if len(tok)>=2: result.append(tok)
    return ' '.join(result)

def preprocess_P4_EPH(t):   return _base(t,True,True,True,True)
EPH_full         = preprocess_P4_EPH
def EPH_no_emoticon(t):  return _base(t,False,True,True,True)
def EPH_no_negation(t):  return _base(t,True,False,True,True)
def EPH_no_banglish(t):  return _base(t,True,True,False,True)
def EPH_no_elongation(t):return _base(t,True,True,True,False)


# ── Paths and input contract ─────────────────────────────────
from pathlib import Path
import hashlib
import json
import pandas as pd

AI_FILE = Path('./youtube_llm_annotated.csv')
PROC_BASE = Path('.')
RES_BASE = Path('.')
AI_LIMIT = 101_000
# Existing gold files are assumed to contain text processed with the
# annotation.ipynb master EPH. Do not apply EPH to them a second time.
GOLD_FILES = {s: PROC_BASE / f'gold_{s}.csv' for s in ('train', 'val', 'test')}
COMBINED_FILE = PROC_BASE / 'combined_train.csv'

def load_dataframe(path):
    df = pd.read_csv(path, low_memory=False, dtype={'text': 'string'})
    missing = {'text', 'label'} - set(df.columns)
    if missing:
        raise ValueError(f'{path}: missing columns {sorted(missing)}')
    df = df[['text', 'label']].copy()
    labels = pd.to_numeric(df['label'], errors='raise')
    if not labels.isin([0, 1]).all():
        raise ValueError(f'{path}: labels must be exactly 0 or 1 (no missing/fractional values)')
    if df.empty or df['text'].isna().any() or df['text'].str.strip().eq('').any():
        raise ValueError(f'{path}: empty dataset or missing/blank text; repair the source split')
    df['text'] = df['text'].astype(str)
    df['label'] = labels.astype('int64')
    return df.reset_index(drop=True)

def text_keys(df):
    return df['text'].str.strip()

def check_disjoint(splits):
    names = list(splits)
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            overlap = set(text_keys(splits[a])) & set(text_keys(splits[b]))
            if overlap:
                raise ValueError(f'{a}/{b}: {len(overlap):,} overlapping texts; repair splits before training')

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

# Read at most one extra row to detect the wrong (larger) source file.
# Never silently take an arbitrary prefix of the full collected dataset.
df_ai = pd.read_csv(AI_FILE, nrows=AI_LIMIT + 1, low_memory=False,
                    dtype={'text': 'string'})
if len(df_ai) != AI_LIMIT:
    raise ValueError(f'Use the exact {AI_LIMIT:,}-row annotation export. '
                     f'Read {len(df_ai):,} rows (capped at {AI_LIMIT + 1:,}).')
missing = {'text', 'ai_label', 'ai_confidence'} - set(df_ai.columns)
if missing:
    raise ValueError(f'{AI_FILE}: missing columns {sorted(missing)}')

df_ai['ai_label'] = pd.to_numeric(df_ai['ai_label'], errors='coerce')
df_ai['ai_confidence'] = df_ai['ai_confidence'].astype('string').str.strip().str.upper()
print(f'AI annotated rows: {len(df_ai):,}')
print('Label distribution:', df_ai['ai_label'].value_counts(dropna=False).to_dict())
print('Confidence:', df_ai['ai_confidence'].value_counts(dropna=False).to_dict())

gold = {s: load_dataframe(p) for s, p in GOLD_FILES.items()}
check_disjoint(gold)
print('Gold split sizes:', {s: len(df) for s, df in gold.items()})

df_pseudo = df_ai.loc[
    df_ai['ai_confidence'].eq('HIGH') & df_ai['ai_label'].isin([0, 1]),
    ['text', 'ai_label']
].rename(columns={'ai_label': 'label'}).copy()
audit = {'ai_rows': len(df_ai), 'high_confidence_valid': len(df_pseudo)}
df_pseudo['label'] = df_pseudo['label'].astype('int64')
print(f"High-confidence valid labels: {len(df_pseudo):,}")
print('Applying original annotation.ipynb EPH...')
df_pseudo['text'] = df_pseudo['text'].fillna('').apply(preprocess_P4_EPH)
before = len(df_pseudo)
df_pseudo = df_pseudo.loc[df_pseudo['text'].str.strip().ne('')].copy()
audit['empty_removed'] = before - len(df_pseudo)

# Match in the processed-text space; gold labels take precedence.
gold_texts = set().union(*(set(text_keys(df)) for df in gold.values()))
before = len(df_pseudo)
df_pseudo = df_pseudo.loc[~text_keys(df_pseudo).isin(gold_texts)].copy()
audit['gold_overlap_removed'] = before - len(df_pseudo)

# Discard conflicting pseudo-label groups, then collapse identical texts.
conflicts = df_pseudo.groupby('text')['label'].nunique()
before = len(df_pseudo)
df_pseudo = df_pseudo.loc[~df_pseudo['text'].isin(conflicts[conflicts > 1].index)].copy()
audit['conflicting_rows_removed'] = before - len(df_pseudo)
before = len(df_pseudo)
df_pseudo = df_pseudo.drop_duplicates('text').reset_index(drop=True)
audit['duplicate_rows_removed'] = before - len(df_pseudo)
audit['pseudo_train_rows'] = len(df_pseudo)
if df_pseudo.empty:
    raise ValueError('No pseudo-labels remain after filtering; inspect the annotation export')

df_combined_train = pd.concat([
    gold['train'].assign(source='human_gold'),
    df_pseudo.assign(source='ai_pseudo')
], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
check_disjoint({'combined_train': df_combined_train,
                'val': gold['val'], 'test': gold['test']})
PROC_BASE.mkdir(parents=True, exist_ok=True)
RES_BASE.mkdir(parents=True, exist_ok=True)
df_combined_train.to_csv(COMBINED_FILE, index=False)
audit['combined_train_rows'] = len(df_combined_train)
audit['eph_source'] = 'annotation.ipynb, notebook cell 8, MASTER SETUP'
audit['gold_preprocessing'] = 'Assumed already processed with the same EPH; provenance not verified'
audit['files'] = {str(p): file_sha256(p) for p in [AI_FILE, *GOLD_FILES.values(), COMBINED_FILE]}
(RES_BASE / 'dataset_audit.json').write_text(json.dumps(audit, indent=2), encoding='utf-8')
print(json.dumps(audit, indent=2))
print(f'Saved {COMBINED_FILE}; validation/test remain the existing human-gold splits.')


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
AI annotated rows: 101,000
Label distribution: {0: 73948, 1: 26855, -1: 197}
Confidence: {'HIGH': 96071, 'LOW': 4732, 'FAILED': 197}


FileNotFoundError: [Errno 2] No such file or directory: 'BanglishDetect\\data\\processed\\gold_train.csv'